# Hybrid Content-Based Movie Recommendation System

**Final modelling notebook — TMDB + Indian cinema**

This notebook loads the prepared, enriched catalogue and builds a multi-block TF-IDF content-based recommender.

**Feature blocks:** genres, keywords, plot text, cast, director, and title.  
**Ranking:** weighted cosine similarity, optional language preference, quality adjustment, and MMR diversity re-ranking.

> This notebook focuses on modelling. Raw-data cleaning, IMDb matching, Wikipedia extraction, and enrichment experiments belong in separate data-preparation scripts/notebooks.
> **Revision 2 (corrected).** Fixes since the first final version: (1) genre labels are normalised to one casing so Indian and TMDB films share a genre vocabulary; (2) `combine()` no longer inflates scores of films with sparse metadata (`missing_penalty`); (3) the Bayesian quality prior has a vote floor; (4) title features no longer vanish for stop-word titles such as *Up* or *Nine*; (5) duplicate, dead and debugging cells removed; (6) a `verbose` switch silences per-query printing. **All outputs were cleared: re-run top to bottom and regenerate the baseline.**


## 1. Imports and configuration

Run the notebook from the project root (`F:/AIML/Movie-Recommendation-System`) so the relative dataset path resolves.

In [1]:
import difflib
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import normalize

RANDOM_STATE = 42
DATA_PATH = Path("movie_project/data/final/movies_unified_enriched_v4.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Run this notebook from the project root."
    )

print("Dataset:", DATA_PATH.resolve())
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | scikit-learn {sklearn.__version__}")


## 2. Load and validate the enriched catalogue

The input is the unified TMDB + Indian catalogue enriched with the latest Indian metadata (v4). The original source CSVs are not modified. The notebook works on an in-memory copy.

Place `movies_unified_enriched_v4.csv` in `movie_project/data/final/` before running this notebook.

In [2]:
movies_raw = pd.read_csv(DATA_PATH, low_memory=False)

required_columns = [
    "movie_id", "source_primary", "imdb_id", "title", "year",
    "genres", "genres_recommender", "keywords", "overview", "tagline",
    "director", "directors", "directors_recommender", "cast_top10",
    "cast_recommender", "languages_available", "primary_language",
    "rating", "votes", "runtime", "recsys_query_ok",
    "recsys_candidate_default"
]

missing_columns = [c for c in required_columns if c not in movies_raw.columns]
if missing_columns:
    raise ValueError(f"Required columns missing from catalogue: {missing_columns}")

if movies_raw["movie_id"].duplicated().any():
    raise ValueError("Duplicate movie_id values found. Resolve duplicates before modelling.")

print("Catalogue:", DATA_PATH.resolve())
print("Catalogue shape:", movies_raw.shape)
print("\nMovie sources:")
display(movies_raw["source_primary"].value_counts(dropna=False).to_frame("records"))
print("\nDuplicate movie IDs:", movies_raw["movie_id"].duplicated().sum())
print("Duplicate non-null IMDb IDs:", movies_raw["imdb_id"].dropna().duplicated().sum())

## 3. Prepare modelling features

The unified catalogue retains its original fields. For recommendation features, it uses `genres_recommender`, `directors_recommender`, and `cast_recommender`, which incorporate the latest Indian enrichment where available and fall back to the unified source metadata otherwise. Plot text and keywords are used only where present; missing text is not invented.

**Genre normalisation:** TMDB labels are Title Case (`Science Fiction`) while Indian labels are lowercase (`comedy`). They are mapped to one canonical form so both sources share a single genre vocabulary.

In [3]:
def split_pipe(value):
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split("|") if x.strip()]

GENRE_SYNONYMS = {
    "sci-fi": "Science Fiction", "scifi": "Science Fiction", "sci fi": "Science Fiction",
    "science fiction": "Science Fiction", "musical": "Music", "romcom": "Romance",
    "film-noir": "Film Noir", "film noir": "Film Noir", "tv movie": "TV Movie",
}

def canon_genre(g):
    """One canonical casing for genre labels from both sources."""
    key = re.sub(r"\s+", " ", str(g).strip().lower())
    return GENRE_SYNONYMS.get(key, key.title())

def norm_token(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def parse_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y"}

movies = movies_raw.copy()

movies["source"] = movies["source_primary"].map({
    "tmdb": "TMDB",
    "imdb_indian_csv": "Indian"
}).fillna(movies["source_primary"].astype(str))

# Convert pipe-separated catalogue fields into token lists.
movies["genres"] = movies["genres_recommender"].apply(
    lambda v: sorted({canon_genre(g) for g in split_pipe(v)})
)
movies["keywords"] = movies["keywords"].apply(split_pipe)
movies["cast_display"] = movies["cast_recommender"].apply(split_pipe)
movies["directors_list"] = movies["directors_recommender"].apply(split_pipe)
movies["languages"] = movies["languages_available"].apply(split_pipe)

# Numeric fields
for col in ["year", "rating", "votes", "runtime"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

# Text and token features
movies["text"] = (
    movies["overview"].fillna("").astype(str) + " " +
    movies["tagline"].fillna("").astype(str)
).str.strip()

movies["keyword_text"] = movies["keywords"].apply(lambda x: " ".join(x))
movies["cast_tok"] = movies["cast_display"].apply(
    lambda names: [norm_token(x) for x in names if norm_token(x)]
)
movies["director_tok"] = movies["directors_list"].apply(
    lambda names: [norm_token(x) for x in names if norm_token(x)]
)
movies["title_clean"] = (
    movies["title"].fillna("").astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Title tokens: drop stop words, but keep all tokens when the title is made only
# of stop words (e.g. "Up", "The One", "Nine") so the film still gets a title vector.
def title_tokens(t):
    toks = t.split()
    kept = [w for w in toks if w not in ENGLISH_STOP_WORDS]
    return kept or toks

movies["title_tok"] = movies["title_clean"].apply(title_tokens)

# Bayesian weighted rating, ranked within each source to avoid comparing
# raw IMDb and TMDB rating scales directly.
MIN_VOTES_PRIOR = 50
movies["quality"] = 0.0
for source, group in movies.groupby("source"):
    valid_votes = group.loc[group["votes"] > 0, "votes"]
    C = group["rating"].mean()
    # prior weight m: 60th percentile of votes, floored so a handful of votes
    # cannot earn a top quality percentile
    m = max(valid_votes.quantile(0.60), MIN_VOTES_PRIOR) if len(valid_votes) else float(MIN_VOTES_PRIOR)

    R = group["rating"].fillna(C if pd.notna(C) else 0.0)
    v = group["votes"].fillna(0.0)
    denom = v + m

    weighted_rating = np.where(
        denom > 0,
        (v / np.where(denom > 0, denom, 1)) * R +
        (m / np.where(denom > 0, denom, 1)) * (C if pd.notna(C) else 0.0),
        R
    )
    movies.loc[group.index, "quality"] = pd.Series(
        weighted_rating, index=group.index
    ).rank(pct=True).fillna(0.0)

# Explicit eligibility flags from the validated catalogue.
movies["query_ok"] = movies["recsys_query_ok"].apply(parse_bool)
movies["eligible"] = movies["recsys_candidate_default"].apply(parse_bool)

# Keep the known invalid placeholder out of recommendation candidates.
invalid_title = movies["title"].fillna("").str.strip().str.casefold().eq(
    "404: error not found"
)
movies.loc[invalid_title, "eligible"] = False

movies["uid"] = np.arange(len(movies))

print(f"Loaded {len(movies):,} movies")
print(movies["source"].value_counts().to_string())
print(f"Valid query records: {movies['query_ok'].sum():,}")
print(f"Eligible recommendation candidates: {movies['eligible'].sum():,}")
print(f"Distinct language labels: {len({x for row in movies['languages'] for x in row})}")

Loaded 50,403 movies
source
Indian    45600
TMDB       4803
Valid query records: 41,969
Eligible recommendation candidates: 21,146
Distinct language labels: 94


## 4. Data coverage and exploratory checks

Coverage is reported separately by source. A missing feature is not treated as evidence that the movie has no such attribute.

In [4]:
coverage_rows = []
for source, group in movies.groupby("source"):
    for label, column in [
        ("Genres", "genres"),
        ("Keywords", "keywords"),
        ("Plot / overview text", "text"),
        ("Cast", "cast_display"),
        ("Director", "directors_list"),
    ]:
        count = group[column].apply(lambda x: bool(x) if isinstance(x, list) else bool(str(x).strip())).sum()
        coverage_rows.append({
            "Source": source,
            "Feature": label,
            "Available": int(count),
            "Total": len(group),
            "Coverage (%)": round(100 * count / len(group), 2) if len(group) else 0
        })

coverage = pd.DataFrame(coverage_rows)
display(coverage.pivot(index="Feature", columns="Source", values="Coverage (%)"))

print("\nIndian metadata coverage:")
display(coverage[coverage["Source"] == "Indian"].reset_index(drop=True))

Source,Indian,TMDB
Feature,,
Cast,88.90,99.10
Director,87.09,99.38
Genres,82.56,99.42
Keywords,0.00,91.42
Plot / overview text,0.00,99.92



Indian metadata coverage:


,Source,Feature,Available,Total,Coverage (%)
0,Indian,Genres,37647,45600,82.56
1,Indian,Keywords,0,45600,0.00
2,Indian,Plot / overview text,0,45600,0.00
3,Indian,Cast,40539,45600,88.90
4,Indian,Director,39712,45600,87.09


In [5]:
print("Top languages:")
display(
    pd.Series([lang for row in movies["languages"] for lang in row])
    .value_counts().head(20).to_frame("movie_count")
)

print("\nSample prepared records:")
display(
    movies[["title", "year", "source", "genres", "directors_list",
            "cast_display", "languages", "rating", "quality", "eligible"]]
    .sample(min(8, len(movies)), random_state=RANDOM_STATE)
)

Top languages:


,movie_count
Hindi,14374
Tamil,6002
Malayalam,5354
Telugu,5296
English,4485
Bengali,4198
Kannada,3233
Marathi,2300
Urdu,1947
Punjabi,1801



Sample prepared records:


,title,year,source,genres,directors_list,cast_display,languages,rating,quality,eligible
14169,Nargis,1992.0,Indian,"[Music, Romance]",[Nazrul Islam],"[Firdous Jamal, Shahida Mini, Madiha Shah, Jaw...","[Punjabi, Urdu]",NaN,0.473279,False
27492,Atithi,1973.0,Indian,"[Drama, Romance]",[Azizur Rahman],"[Razzak, Shabana, Alamgir, Syed Hasan Imam, In...",[Bengali],NaN,0.473279,False
9430,Ahetuk,2015.0,Indian,"[Drama, Romance, Thriller]",[Bani Das],"[Gunjan Bhardwaj, Amrita Gogoi, Tapan Das, Pri...",[Assamese],7.4,0.843629,True
7096,Aval Viswathayayirunnu,1978.0,Indian,[Drama],[Jeassy],"[M.G. Soman, Vincent, Jayabharathi, Jose Praka...",[Malayalam],NaN,0.473279,False
46601,Agnisakshi,2003.0,Indian,[Drama],[Jadumoni Dutta],"[Zerifa Wahid, Jatin Bora, Jadumoni Dutta, Bij...",[Assamese],NaN,0.473279,False
3767,Secretary,2002.0,TMDB,"[Comedy, Drama, Romance]",[Steven Shainberg],"[James Spader, Maggie Gyllenhaal, Jeremy Davie...",[English],6.7,0.762857,True
35118,Rafoo Chakkar: Fun on the Run,2008.0,Indian,"[Comedy, Crime, Drama]",[B.H. Tharun Kumar],"[Nauheed Cyrusi, Aslam Khan, Nisha Rawal, Yudh...",[Hindi],3.3,0.028015,True
49227,Bagh bandi khela,2018.0,Indian,[Thriller],"[Haranath Chakraborty, Raja Chanda, Sujit Mondal]","[Sayantika Banerjee, Ayan Bhattacharjee, Saura...",[Bengali],4.6,0.030746,True


## 5. Build separate TF-IDF feature blocks

Each feature family is vectorised independently. This keeps the scoring interpretable and allows feature weights to be adjusted at recommendation time.

In [6]:
def identity_analyzer(tokens):
    return tokens

vectorizers = {
    "genres": TfidfVectorizer(analyzer=identity_analyzer),
    "keywords": TfidfVectorizer(
        stop_words="english", min_df=2, sublinear_tf=True
    ),
    "text": TfidfVectorizer(
        stop_words="english", ngram_range=(1, 2),
        min_df=2, max_features=40000, sublinear_tf=True
    ),
    "cast": TfidfVectorizer(analyzer=identity_analyzer),
    "director": TfidfVectorizer(analyzer=identity_analyzer),
    "title": TfidfVectorizer(analyzer=identity_analyzer),
}

feature_inputs = {
    "genres": movies["genres"].tolist(),
    "keywords": movies["keyword_text"].tolist(),
    "text": movies["text"].tolist(),
    "cast": movies["cast_tok"].tolist(),
    "director": movies["director_tok"].tolist(),
    "title": movies["title_tok"].tolist(),
}

X = {
    block: vectorizers[block].fit_transform(feature_inputs[block])
    .tocsr().astype(np.float32)
    for block in vectorizers
}

feature_report = pd.DataFrame([
    {
        "Feature block": block,
        "Vocabulary size": matrix.shape[1],
        "Movies with data": int((matrix.getnnz(axis=1) > 0).sum())
    }
    for block, matrix in X.items()
])
display(feature_report)
# Sanity check: no genre label may exist in two casings.
genre_vocab = sorted(vectorizers["genres"].vocabulary_)
dupes = {g.lower() for g in genre_vocab if sum(h.lower() == g.lower() for h in genre_vocab) > 1}
assert not dupes, f"Genre labels in multiple casings: {dupes}"
print(f"Genre vocabulary ({len(genre_vocab)}):", genre_vocab)


,Feature block,Vocabulary size,Movies with data
0,genres,23,42422
1,keywords,4062,4379
2,text,19059,4799
3,cast,71323,45299
4,director,17962,44485
5,title,37782,50403


Genre vocabulary (23): ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History', 'Horror', 'Music', 'Mystery', 'Reality-Tv', 'Romance', 'Science Fiction', 'Sport', 'TV Movie', 'Thriller', 'War', 'Western']


In [23]:
# Cross-source genre sharing

indian_genres = set()
tmdb_genres = set()

for source, genres in zip(movies["source"], movies["genres"]):
    if source == "Indian":
        indian_genres.update(genres)
    elif source == "TMDB":
        tmdb_genres.update(genres)

shared_genres = indian_genres & tmdb_genres

print("Indian unique genres:", len(indian_genres))
print("TMDB unique genres:", len(tmdb_genres))
print("Shared genres:", len(shared_genres))
print("Shared genre names:", sorted(shared_genres))

print("\nIndian-only genres:", sorted(indian_genres - tmdb_genres))
print("TMDB-only genres:", sorted(tmdb_genres - indian_genres))

Indian unique genres: 21
TMDB unique genres: 20
Shared genres: 18
Shared genre names: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'Thriller', 'War', 'Western']

Indian-only genres: ['Biography', 'Reality-Tv', 'Sport']
TMDB-only genres: ['Foreign', 'TV Movie']


## 6. Recommendation engine

The ranking score combines weighted cosine similarity across the available feature blocks. Optional language preference, quality adjustment, and MMR diversification are applied after content similarity.

**Sparse-metadata handling:** a candidate missing a feature block counts `missing_penalty` of that block's weight in the denominator (default 0.5). `0` reproduces the old behaviour (films with little data can outscore well-documented ones); `1` fully penalises missing blocks.

In [7]:
DEFAULT_WEIGHTS = {"genres": 3.0, "keywords": 2.0, "text": 1.5, "cast": 1.5, "director": 1.0, "title": 0.5, "year": 0.75}
# "What matters most to you?"  -> ready-made weight sets (usable as  weights=PRESETS["story_first"])
PRESETS = {
    "balanced":        DEFAULT_WEIGHTS,
    "genre_first":     {"genres": 6.0, "keywords": 1.5, "text": 1.0, "cast": 1.0, "director": 0.5, "title": 0.5, "year": 0.75},
    "story_first":     {"genres": 2.0, "keywords": 3.5, "text": 3.0, "cast": 0.75, "director": 0.5, "title": 0.5, "year": 0.5},
    "cast_crew_first": {"genres": 2.0, "keywords": 1.0, "text": 1.0, "cast": 3.0, "director": 2.5, "title": 0.5, "year": 0.5},
}
LANGUAGE_MODES  = {"ignore": 0.0, "balanced": 0.15, "prefer": 0.6, "strict": 0.0}   # strict = hard filter on top


class MovieRecommender:
    def __init__(self, movies, X, vecs, weights=DEFAULT_WEIGHTS, year_scale=12.0, missing_penalty=0.5):
        self.m, self.X, self.vecs, self.weights, self.tau = movies.reset_index(drop=True), X, vecs, dict(weights), year_scale
        self.missing_penalty = missing_penalty
        self.N        = len(self.m)
        self.year     = self.m.year.to_numpy(float)
        self.quality  = self.m.quality.to_numpy(float)
        self.eligible = self.m.eligible.to_numpy(bool)
        # Track which movies have usable data in each feature block
        self.block_available = {
            b: np.asarray(X[b].getnnz(axis=1) > 0)
            for b in X
        }

        # Year is available only when it is not missing
        self.block_available["year"] = np.isfinite(self.year)
        self.keys = self.m.movie_id.to_numpy()
        self.titles_lc = self.m.title.fillna("").str.lower().to_numpy()

        # multi-hot language matrix  (a film dubbed in 3 languages matches all 3)
        langs = sorted({l for ls in self.m.languages for l in ls})
        self.lang_idx = {l: i for i, l in enumerate(langs)}
        language_pairs = [(i, self.lang_idx[l]) for i, ls in enumerate(self.m.languages) for l in ls]
        if language_pairs:
            r, c = zip(*language_pairs)
            self.L = sparse.csc_matrix(
                (np.ones(len(r), dtype=bool), (r, c)),
                shape=(self.N, len(langs))
            )
        else:
            self.L = sparse.csc_matrix((self.N, len(langs)), dtype=bool)
        self._lm = {}

        # binary "has this token" matrices, used for preference queries ("I want Sci-Fi AND Leonardo DiCaprio")
        self.B = {b: (X[b] > 0).astype(np.float32).tocsr() for b in ["genres", "cast", "director", "keywords"]}

        # space used for the diversity (MMR) step
        self.Xdiv = normalize(sparse.hstack([X["genres"], X["keywords"], X["text"], X["cast"]]).tocsr())

        # people lookup (display name by normalised token)
        self.people = {}
        for cast, director_list in zip(
            self.m.cast_display,
            self.m.directors_list
        ):

            for p in cast + director_list:
                token = norm_token(p)
                if token:
                    self.people[token] = p

    # ---------------------------------------------------------------- helpers
    def languages(self):
        return sorted(self.lang_idx)

    def lang_mask(self, language):
        language = language.strip().title()
        if language not in self.lang_idx:
            raise KeyError(f"Unknown language '{language}'. Available: {', '.join(self.languages())}")
        if language not in self._lm:
            self._lm[language] = self.L[:, self.lang_idx[language]].toarray().ravel()
        return self._lm[language]

    def find_movie(self, title, year=None, language=None):
        t = title.lower().strip()
        mask = self.titles_lc == t
        if year is not None:     mask &= (self.year == year)
        if language is not None: mask &= self.lang_mask(language)
        cand = np.flatnonzero(mask)
        if len(cand) == 0:
            close = difflib.get_close_matches(t, set(self.titles_lc), n=5, cutoff=0.7)
            sugg = [self.m.title[np.flatnonzero(self.titles_lc == c)[0]] for c in close]
            raise KeyError(f"'{title}' not found. Did you mean: {sugg}")
        if len(cand) > 1:
            cand = cand[np.argsort(-self.m.votes.to_numpy()[cand])]
            others = [
                f"{self.m.title.iloc[j]} "
                f"({int(self.year[j]) if not np.isnan(self.year[j]) else '?'}, "
                f"{self.m.primary_language.iloc[j]})"
                for j in cand[1:4]
            ]
            print(f"ℹ︎ {len(cand)} movies titled '{title}' – using the most-voted one. Others: {others}. Use year= / language= to choose.")
        return int(cand[0])

    def find_person(self, fragment, limit=8):
        f = norm_token(fragment)
        return [self.people[t] for t in self.people if f in t][:limit]

    # ---------------------------------------------------------------- similarity
    def block_sims(self, q, q_year=None, coverage=None):
        # cosine per block. For preference queries `coverage` = {block: alpha} blends in "share of requested tokens present":
        # a movie with Sci-Fi + Action + Thriller still fully satisfies a request for "Sci-Fi" (plain cosine would punish the extras).
        coverage, sims = coverage or {}, {}
        for b, qb in q.items():
            if qb is None or qb.nnz == 0: continue
            s = (self.X[b] @ qb.T).toarray().ravel()
            if b in coverage:
                qbin = (qb > 0).astype(np.float32)
                cov = (self.B[b] @ qbin.T).toarray().ravel() / qbin.sum()
                s = coverage[b] * cov + (1 - coverage[b]) * s
            sims[b] = s
        if q_year is not None and not np.isnan(q_year):
            sims["year"] = np.nan_to_num(np.exp(-np.abs(self.year - q_year) / self.tau))
        return sims

    def query_sims(self, i):
        return self.block_sims({b: self.X[b][i] for b in self.X}, self.year[i])


    def combine(self, sims, weights=None):
        w = weights or self.weights

        num = np.zeros(self.N, dtype=float)
        den = np.zeros(self.N, dtype=float)

        for b, s in sims.items():
            weight = w.get(b, 0)

            if weight <= 0:
                continue

            available = self.block_available[b]

            num += weight * np.nan_to_num(s, nan=0.0)
            # A block the query uses but the candidate lacks still counts
            # missing_penalty * weight, so sparse films cannot win on 2-3 blocks.
            den += weight * (available + self.missing_penalty * (~available))

        return np.divide(
            num,
            den,
            out=np.zeros(self.N, dtype=float),
            where=den > 0
        )


    # ---------------------------------------------------------------- ranking
    def select(self, score, exclude_key=None, n=10, mask=None):
        s = score.astype(float).copy()

        s[~self.eligible] = -np.inf

        if mask is not None:
            s[~mask] = -np.inf

        k = min(n * 8 + 50, self.N)

        if k == 0:
            return np.array([], dtype=int)

        order = np.argpartition(-s, k - 1)[:k]
        order = order[np.argsort(-s[order])]

        out = []
        seen = {exclude_key}

        for j in order:
            if not np.isfinite(s[j]):
                break

            if self.keys[j] in seen:
                continue

            seen.add(self.keys[j])
            out.append(j)

            if len(out) == n:
                break

        return np.array(out, dtype=int)

    def _mmr(self, cand, score, n, lam):
        M = self.Xdiv[cand]; S = (M @ M.T).toarray()
        sel, rem = [], list(range(len(cand)))
        while rem and len(sel) < n:
            best = max(rem, key=lambda r: lam * score[cand[r]] - (1 - lam) * (max(S[r][s] for s in sel) if sel else 0))
            sel.append(best); rem.remove(best)
        return cand[sel]

    def rank(self, sims, qi=None, n=10, weights=None, lang_target=None,
         lang_bonus=0.15, strict=False, quality_weight=0.1,
         diversify=False, mmr_lambda=0.7, extra_mask=None):

        score = self.combine(sims, weights)

        # Exclude movies with zero content similarity
        mask = score > 0

        if extra_mask is not None:
            mask &= extra_mask

        if lang_target is not None:
            lm = self.lang_mask(lang_target)
            score = score + lang_bonus * lm

            if strict:
                mask &= lm

        if quality_weight:
            score = score + quality_weight * self.quality

        excl = self.keys[qi] if qi is not None else None

        idx = self.select(
            score,
            excl,
            n * 4 if diversify else n,
            mask
        )

        if diversify and len(idx) > n:
            idx = self._mmr(idx, score, n, mmr_lambda)

        return idx, score[idx]

    # ---------------------------------------------------------------- explanations & display
    def _why_movie(self, qi, j):
        a, b, r = self.m.iloc[qi], self.m.iloc[j], []
        sg = [g for g in a.genres if g in b.genres]
        if sg: r.append("genre: " + "/".join(sg))
        shared_languages = set(a.languages) & set(b.languages)
        if shared_languages: r.append("language: " + ", ".join(sorted(shared_languages)[:2]))
        shared_directors = (
            set(a.directors_list) & set(b.directors_list)
        )

        if shared_directors:
            r.append(
                "director: " + ", ".join(sorted(shared_directors))
            )
        sc = [c for c in a.cast_display if c in b.cast_display]
        if sc: r.append("cast: " + ", ".join(sc[:2]))
        sk = [k for k in a.keywords if k in b.keywords]
        if sk: r.append("themes: " + ", ".join(sk[:3]))
        if not np.isnan(a.year) and not np.isnan(b.year) and abs(a.year - b.year) <= 3: r.append("same era")
        return "; ".join(r)

    def _present(self, idx, scores, why):
        m = self.m.iloc[idx]

        out = pd.DataFrame({
            "Movie_ID": m.movie_id.values,
            "Title": m.title.values,
            "Year": m.year.astype("Int64").values,
            "Language": m.languages.apply(", ".join).values,
            "Genres": m.genres.apply(", ".join).values,
            "Director": m.directors_list.apply(
                lambda names: ", ".join(names)
            ).values,
            "Rating": m.rating.round(1).values,
            "Votes": m.votes.astype("Int64").values,
            "Score": np.round(scores, 3),
            "Why": [why(j) for j in idx]
        })

        out.index = range(1, len(out) + 1)
        return out

    # ---------------------------------------------------------------- public API
    def recommend(self, title, n=10, year=None, language=None, language_mode="balanced", quality_weight=0.1,
                  diversify=False, weights=None, source_language=None, verbose=True):
        # title: query movie.  language: target language of the results (default: language of the query movie).
        qi = self.find_movie(title, year, source_language)
        if isinstance(language_mode, str):
            bonus, strict = LANGUAGE_MODES[language_mode], language_mode == "strict"
        else:
            bonus, strict = float(language_mode), False
        use_lang = language_mode != "ignore"
        default_language = self.m.primary_language.iloc[qi]
        if pd.isna(default_language) or not default_language:
            available_languages = self.m.languages.iloc[qi]
            default_language = (
                available_languages[0] if available_languages else None
            )

        target = (language or default_language) if use_lang else None
        q = self.m.iloc[qi]
        if verbose:
            print(f"Because you picked: {q.title} ({'' if np.isnan(q.year) else int(q.year)}) · {', '.join(q.languages)} · {', '.join(q.genres)}"
                  f"   |  language_mode={language_mode}" + (f", target={target}" if target else ""))
        idx, sc = self.rank(self.query_sims(qi), qi, n, weights, target, bonus, strict, quality_weight, diversify)
        return self._present(idx, sc, lambda j: self._why_movie(qi, j))

    def recommend_by_preferences(self, genres=None, language=None, actors=None, director=None, keywords=None,
                                 plot=None, year=None, n=10, language_mode="prefer", quality_weight=0.1,
                                 diversify=False, weights=None):
        # cold-start: no movie needed, describe what you want
        q, notes = {}, []
        if genres:
            gs = [canon_genre(g) for g in genres]
            bad = [g for g in gs if g not in self.vecs["genres"].vocabulary_]
            if bad: notes.append(f"unknown genre(s) ignored: {bad}. Available: {sorted(self.vecs['genres'].vocabulary_)}")
            q["genres"] = self.vecs["genres"].transform([[g for g in gs if g not in bad]])
        if actors:
            toks = [norm_token(a) for a in actors]
            bad = [a for a, t in zip(actors, toks) if t not in self.vecs["cast"].vocabulary_]
            if bad: notes.append(f"actor(s) not in data: {bad} (try .find_person('name'))")
            q["cast"] = self.vecs["cast"].transform([[t for t in toks if t in self.vecs["cast"].vocabulary_]])
        if director:
            t = norm_token(director)
            if t in self.vecs["director"].vocabulary_: q["director"] = self.vecs["director"].transform([[t]])
            else: notes.append(f"director not in data: {director} (try .find_person('name'))")
        if keywords: q["keywords"] = self.vecs["keywords"].transform([" ".join(keywords)])
        if plot:     q["text"]     = self.vecs["text"].transform([plot.lower()])
        for nt in notes: print("⚠", nt)

        if isinstance(language_mode, str):
            bonus, strict = LANGUAGE_MODES[language_mode], language_mode == "strict"
        else:
            bonus, strict = float(language_mode), False
        target = language if (language and language_mode != "ignore") else None
        sims = self.block_sims(q, year, coverage={"genres": 0.7, "keywords": 0.7, "cast": 1.0, "director": 1.0})
        idx, sc = self.rank(sims, None, n, weights, target, bonus, strict, quality_weight, diversify)

        want_g, want_a = set(gs) if genres else set(), {norm_token(a) for a in actors or []}
        def why(j):
            r, mj = [], self.m.iloc[j]
            sg = [g for g in mj.genres if g in want_g]
            if sg: r.append("genre: " + "/".join(sg))
            if language and language.title() in mj.languages: r.append(f"language: {language.title()}")
            sa = [a for a, t in zip(mj.cast_display, mj.cast_tok) if t in want_a]
            if sa: r.append("cast: " + ", ".join(sa))
            if director and norm_token(director) in mj.director_tok: r.append(f"director: {mj.director}")
            if not r and plot: r.append("plot / theme similarity")
            return "; ".join(r)
        return self._present(idx, sc, why)


rec = MovieRecommender(movies, X, vectorizers)
print(f"Recommender ready · {rec.N:,} movies · {len(rec.languages())} languages · missing_penalty={rec.missing_penalty}")

Recommender ready · 50,403 movies · 94 languages · missing_penalty=0.5


## 7. Recommendation examples

Try both international and Indian movie titles. If a title is ambiguous, pass its year or language.

In [8]:
demo_queries = [
    ("Avatar", 2009),
    ("Inception", 2010),
    ("The Dark Knight", 2008),
    ("3 Idiots", None),
    ("Ahetuk", 2015),
    ("Classmates", 2006),
]

for title, year in demo_queries:
    print("\n" + "=" * 72)
    print(f"QUERY: {title}" + (f" ({year})" if year else ""))
    try:
        display(rec.recommend(
            title=title,
            year=year,
            n=10,
            language_mode="ignore",
            diversify=False
        ))
    except KeyError as exc:
        print(exc)


QUERY: Avatar (2009)
Because you picked: Avatar (2009) · English, Spanish · Action, Adventure, Fantasy, Science Fiction   |  language_mode=ignore


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,tmdb:54138,Star Trek Into Darkness,2013,English,"Action, Adventure, Science Fiction",J.J. Abrams,7.4,4418,0.480,genre: Action/Adventure/Science Fiction; langu...
2,imdb:tt1349460,Uncharted 2: Among Thieves,2009,Nepali,"Action, Adventure, Fantasy","Amy Hennig, Bruce Straley",9.4,21467,0.470,genre: Action/Adventure/Fantasy; same era
3,tmdb:13475,Star Trek,2009,English,"Action, Adventure, Science Fiction",J.J. Abrams,7.4,4518,0.466,genre: Action/Adventure/Science Fiction; langu...
4,tmdb:127585,X-Men: Days of Future Past,2014,English,"Action, Adventure, Fantasy, Science Fiction",Bryan Singer,7.5,6032,0.452,genre: Action/Adventure/Fantasy/Science Fictio...
5,tmdb:2756,The Abyss,1989,English,"Action, Adventure, Science Fiction, Thriller",James Cameron,7.1,808,0.437,genre: Action/Adventure/Science Fiction; langu...
6,tmdb:679,Aliens,1986,English,"Action, Horror, Science Fiction, Thriller",James Cameron,7.7,3220,0.437,genre: Action/Science Fiction; language: Engli...
7,tmdb:49521,Man of Steel,2013,English,"Action, Adventure, Fantasy, Science Fiction",Zack Snyder,6.5,6359,0.436,genre: Action/Adventure/Fantasy/Science Fictio...
8,tmdb:118340,Guardians of the Galaxy,2014,English,"Action, Adventure, Science Fiction",James Gunn,7.9,9742,0.436,genre: Action/Adventure/Science Fiction; langu...
9,imdb:tt7946836,Animal World,2018,Hindi,"Action, Fantasy, Science Fiction",Yan Han,6.5,3808,0.435,genre: Action/Fantasy/Science Fiction
10,imdb:tt0366886,Patalghar,2003,Bengali,"Fantasy, Science Fiction",Abhijit Chowdhury,7.2,224,0.432,genre: Fantasy/Science Fiction



QUERY: Inception (2010)
Because you picked: Inception (2010) · English, Japanese, French · Action, Adventure, Mystery, Science Fiction, Thriller   |  language_mode=ignore


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,tmdb:157336,Interstellar,2014,English,"Adventure, Drama, Science Fiction",Christopher Nolan,8.1,10867,0.472,genre: Adventure/Science Fiction; language: En...
2,imdb:tt1821682,Traffic,2011,Malayalam,"Action, Adventure, Mystery",Rajesh Pillai,8.1,4271,0.443,genre: Action/Adventure/Mystery; same era
3,imdb:tt1305797,Enthiran,2010,Tamil,"Action, Science Fiction, Thriller",Shankar Shanmugham,7.1,25248,0.442,genre: Action/Science Fiction/Thriller; same era
4,tmdb:49026,The Dark Knight Rises,2012,English,"Action, Crime, Drama, Thriller",Christopher Nolan,7.6,9106,0.435,genre: Action/Thriller; language: English; dir...
5,imdb:tt5947332,Cryptic Road,2016,Hindi,"Mystery, Science Fiction, Thriller","Moosa Saleem, Hardik Kapadia",7.3,23,0.433,genre: Mystery/Science Fiction/Thriller
6,tmdb:1124,The Prestige,2006,English,"Drama, Mystery, Thriller",Christopher Nolan,8.0,4391,0.433,genre: Mystery/Thriller; language: English; di...
7,imdb:tt3518888,Poovarasam Peepee,2014,Tamil,"Adventure, Mystery, Thriller",Halitha Shameem,7.6,74,0.428,genre: Adventure/Mystery/Thriller
8,imdb:tt4265964,Badshahi Angti,2014,Bengali,"Adventure, Mystery, Thriller",Sandip Ray,6.6,977,0.423,genre: Adventure/Mystery/Thriller
9,imdb:tt2123894,An Human,2011,Tamil,"Drama, Mystery, Science Fiction",Koushik Kr,6.9,22,0.422,genre: Mystery/Science Fiction; same era
10,imdb:tt1725795,7 Aum Arivu,2011,Tamil,"Action, Science Fiction, Thriller",A.R. Murugadoss,6.3,10417,0.421,genre: Action/Science Fiction/Thriller; same era



QUERY: The Dark Knight (2008)
Because you picked: The Dark Knight (2008) · English, Mandarin · Action, Crime, Drama, Thriller   |  language_mode=ignore


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,tmdb:49026,The Dark Knight Rises,2012,English,"Action, Crime, Drama, Thriller",Christopher Nolan,7.6,9106,0.774,genre: Action/Crime/Drama/Thriller; language: ...
2,tmdb:272,Batman Begins,2005,"English, Urdu, Mandarin","Action, Crime, Drama",Christopher Nolan,7.5,7359,0.674,"genre: Action/Crime/Drama; language: English, ..."
3,imdb:tt6805436,Navagraha,2008,Kannada,"Action, Crime, Thriller",Dinakar Thoogudeepa,7.6,106,0.521,genre: Action/Crime/Thriller; same era
4,imdb:tt0973783,Big B,2007,Malayalam,"Action, Crime, Thriller",Amal Neerad,7.6,2835,0.517,genre: Action/Crime/Thriller; same era
5,imdb:tt1451409,Pokkiri,2007,Tamil,"Action, Crime, Thriller",Prabhu Deva,7.2,8853,0.516,genre: Action/Crime/Thriller; same era
6,imdb:tt0920457,Billa,2007,Tamil,"Action, Crime, Thriller",Vishnuvardhan,7.2,6413,0.516,genre: Action/Crime/Thriller; same era
7,imdb:tt1496729,Aaranya Kaandam,2010,Tamil,"Action, Crime, Thriller",Thiagarajan Kumararaja,8.6,3936,0.513,genre: Action/Crime/Thriller; same era
8,imdb:tt0816258,Pokiri,2006,Telugu,"Action, Crime, Thriller",Puri Jagannadh,7.9,12867,0.512,genre: Action/Crime/Thriller; same era
9,imdb:tt0455829,Vettaiyaadu Vilaiyaadu,2006,Tamil,"Action, Crime, Thriller",Gautham Vasudev Menon,7.9,6818,0.512,genre: Action/Crime/Thriller; same era
10,imdb:tt0461936,Don,2006,Hindi,"Action, Crime, Thriller",Farhan Akhtar,7.2,34672,0.509,genre: Action/Crime/Thriller; same era


... (2 outputs left out)


QUERY: Ahetuk (2015)
Because you picked: Ahetuk (2015) · Assamese · Drama, Romance, Thriller   |  language_mode=ignore


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt4373868,Classmates,2015,Marathi,"Drama, Romance, Thriller",Aditya Sarpotdar,7.4,311,0.652,genre: Drama/Romance/Thriller; same era
2,imdb:tt3845746,Njan Steve Lopez,2014,Malayalam,"Drama, Romance, Thriller",Rajeev Ravi,7.4,385,0.643,genre: Drama/Romance/Thriller; same era
3,imdb:tt4799774,Devara Naadalli,2016,Kannada,"Drama, Romance, Thriller",B. Suresha,7.5,28,0.635,genre: Drama/Romance/Thriller; same era
4,imdb:tt6409272,Beautiful Manasugalu,2017,Kannada,"Drama, Romance, Thriller",Jayatheertha,7.7,131,0.634,genre: Drama/Romance/Thriller; same era
5,imdb:tt2476214,Ayalum Njanum Thammil...,2012,Malayalam,"Drama, Romance, Thriller",Lal Jose,8.0,2579,0.630,genre: Drama/Romance/Thriller; same era
6,imdb:tt2404187,Bloody Isshq,2013,Hindi,"Drama, Romance, Thriller",Arup Dutta,7.3,33,0.627,genre: Drama/Romance/Thriller; same era
7,imdb:tt5021536,Kendasampige,2015,Kannada,"Romance, Thriller",Suri,8.3,1863,0.626,genre: Romance/Thriller; same era
8,imdb:tt10443846,Yeh Saali Aashiqui,2019,Hindi,"Drama, Romance, Thriller",Cherag Ruparel,7.9,3050,0.623,genre: Drama/Romance/Thriller
9,imdb:tt9448332,Ishq,2019,Malayalam,"Drama, Romance, Thriller",Anuraj Manohar,7.5,2074,0.622,genre: Drama/Romance/Thriller
10,imdb:tt2976172,Sixteen,2013,Hindi,"Drama, Romance, Thriller","Raj Purohit, LAl Mohan",6.3,1667,0.620,genre: Drama/Romance/Thriller; same era



QUERY: Classmates (2006)
Because you picked: Classmates (2006) · Malayalam · Drama, Mystery, Romance   |  language_mode=ignore


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt1018698,Classmates,2007,Telugu,"Drama, Mystery, Romance",Vijay K. Bhaskar,6.1,60,0.640,genre: Drama/Mystery/Romance; same era
2,imdb:tt3994168,Shehr-e-Zaat,2012,Urdu,"Mystery, Romance",,8.4,513,0.631,genre: Mystery/Romance
3,tmdb:2044,The Lake House,2006,English,"Drama, Mystery, Romance",Alejandro Agresti,6.5,670,0.629,genre: Drama/Mystery/Romance; same era
4,imdb:tt2086879,Megher Koley Rod,2008,Bengali,"Drama, Mystery, Romance",Nargis Akter,6.8,24,0.621,genre: Drama/Mystery/Romance; same era
5,imdb:tt1373156,Karthik Calling Karthik,2010,"Hindi, Urdu","Drama, Mystery, Romance",Vijay Lalwani,7.1,10792,0.620,genre: Drama/Mystery/Romance
6,imdb:tt2123129,3,2012,Tamil,"Drama, Mystery, Romance",Aishwarya Rajinikanth,7.3,4467,0.609,genre: Drama/Mystery/Romance
7,imdb:tt0400186,Chanthupottu,2005,Malayalam,"Comedy, Drama, Romance",Lal Jose,6.6,724,0.591,genre: Drama/Romance; language: Malayalam; dir...
8,tmdb:18615,Fur: An Imaginary Portrait of Diane Arbus,2006,English,"Drama, Mystery, Romance",Steven Shainberg,5.8,73,0.586,genre: Drama/Mystery/Romance; same era
9,imdb:tt2575568,Aadum Koothu,2006,Tamil,"Drama, Mystery",T.V. Chandran,7.2,15,0.585,genre: Drama/Mystery; cast: Jagathy Sreekumar;...
10,imdb:tt4361472,DristiBhram,2015,Nepali,"Drama, Mystery, Romance",Kiran Thapa,7.6,19,0.584,genre: Drama/Mystery/Romance


## 8. Language-aware recommendations

Available modes:

- `ignore`: no language preference.
- `balanced`: small preference for the selected/default language.
- `prefer`: stronger language preference.
- `strict`: exclude candidates that do not match the selected/default language.

You can override the query movie's default language with the `language` argument.

In [9]:
# Example: recommendations for an Indian movie, preferring Hindi.
# Change the title/year if you want to test another movie.
display(rec.recommend(
    title="3 Idiots",
    n=10,
    language="Hindi",
    language_mode="prefer",
    diversify=False
))

Because you picked: 3 Idiots (2009) · Hindi · Comedy, Drama   |  language_mode=prefer, target=Hindi


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt0374887,Munna Bhai M.B.B.S.,2003,Hindi,"Comedy, Drama",Rajkumar Hirani,8.1,74793,1.378,genre: Comedy/Drama; language: Hindi; director...
2,imdb:tt11167150,Mumbai Xpress,2005,Hindi,Comedy,,6.9,58,1.255,genre: Comedy; language: Hindi
3,imdb:tt1434447,Rocket Singh: Salesman of the Year,2009,Hindi,"Comedy, Drama",Shimit Amin,7.5,19102,1.253,genre: Comedy/Drama; language: Hindi; same era
4,imdb:tt1288638,Dasvidaniya,2008,Hindi,"Comedy, Drama",Shashant Shah,7.8,6081,1.245,genre: Comedy/Drama; language: Hindi; same era
5,imdb:tt2338151,PK,2014,Hindi,"Comedy, Drama, Music",Rajkumar Hirani,8.1,165617,1.244,genre: Comedy/Drama; language: Hindi; director...
6,imdb:tt1447508,Peepli [Live],2010,Hindi,"Comedy, Drama","Anusha Rizvi, Mahmood Farooqui",7.4,11885,1.244,genre: Comedy/Drama; language: Hindi; same era
7,imdb:tt1629391,Tere Bin Laden,2010,"Hindi, Punjabi, Urdu","Comedy, Drama",Abhishek Sharma,7.2,10196,1.243,genre: Comedy/Drama; language: Hindi; same era
8,imdb:tt5342784,Le Lotta,2017,Hindi,"Comedy, Drama",Manish Mitra,6.5,15,1.242,genre: Comedy/Drama; language: Hindi
9,imdb:tt1221139,Welcome to Sajjanpur,2008,Hindi,"Comedy, Drama",Shyam Benegal,6.9,3072,1.240,genre: Comedy/Drama; language: Hindi; same era
10,imdb:tt1740670,Dus Tola,2010,Hindi,"Comedy, Drama",Ajoy Varma Raja,7.0,300,1.240,genre: Comedy/Drama; language: Hindi; same era


## 9. Cold-start recommendations from preferences

This mode recommends movies without requiring a seed movie. Supply any combination of genres, actors, director, keywords, plot description, year, and language.

In [10]:
display(rec.recommend_by_preferences(
    genres=["Comedy", "Drama"],
    language="Hindi",
    year=2015,
    n=10,
    language_mode="prefer",
    diversify=True
))

,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt4088588,Rainbow,2015,"Hindi, Rajasthani","Comedy, Drama",Nagesh Kukunoor,7.9,3623,1.699,genre: Comedy/Drama; language: Hindi
2,imdb:tt3767372,Piku,2015,Hindi,"Comedy, Drama",Shoojit Sircar,7.6,29340,1.698,genre: Comedy/Drama; language: Hindi
3,imdb:tt4949324,Mohalla Assi,2015,Hindi,"Comedy, Drama",Chandra Prakash Dwivedi,6.8,1579,1.692,genre: Comedy/Drama; language: Hindi
4,imdb:tt3495030,Dum Laga Ke Haisha,2015,Hindi,"Comedy, Drama, Romance",Sharat Katariya,7.5,18313,1.637,genre: Comedy/Drama; language: Hindi
5,imdb:tt2614722,Umrika,2015,Hindi,"Comedy, Drama",Prashant Nair,6.6,1299,1.689,genre: Comedy/Drama; language: Hindi
6,imdb:tt4921860,MSG 2 the Messenger,2015,"Hindi, Malayalam, Tamil, Telugu","Action, Comedy, Drama",Gurmeet Ram Rahim Singh,6.7,8420,1.635,genre: Comedy/Drama; language: Hindi
7,imdb:tt4990516,You Are My Sunday,2016,Hindi,"Comedy, Drama",Milind Dhaimade,7.8,1316,1.682,genre: Comedy/Drama; language: Hindi
8,imdb:tt5997928,Hotel Salvation,2016,Hindi,"Comedy, Drama",Shubhashish Bhutiani,7.1,1977,1.679,genre: Comedy/Drama; language: Hindi
9,imdb:tt4807830,Lipstick Under My Burkha,2016,Hindi,"Comedy, Drama",Alankrita Shrivastava,6.8,6288,1.676,genre: Comedy/Drama; language: Hindi
10,imdb:tt3614516,Ankhon Dekhi,2013,Hindi,"Comedy, Drama",Rajat Kapoor,8.0,10398,1.668,genre: Comedy/Drama; language: Hindi


## 10. Diversity re-ranking (MMR)

MMR balances relevance against similarity to items already selected. A lower `mmr_lambda` favours more diversity; a higher value favours relevance.

In [11]:
normal_results = rec.recommend(
    title="Inception", n=10,
    language_mode="ignore", diversify=False
)

diverse_results = rec.recommend(
    title="Inception", n=10,
    language_mode="ignore", diversify=True
)

print("Standard ranking:")
display(normal_results)

print("MMR-diversified ranking:")
display(diverse_results)

Because you picked: Inception (2010) · English, Japanese, French · Action, Adventure, Mystery, Science Fiction, Thriller   |  language_mode=ignore
Because you picked: Inception (2010) · English, Japanese, French · Action, Adventure, Mystery, Science Fiction, Thriller   |  language_mode=ignore
Standard ranking:


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,tmdb:157336,Interstellar,2014,English,"Adventure, Drama, Science Fiction",Christopher Nolan,8.1,10867,0.472,genre: Adventure/Science Fiction; language: En...
2,imdb:tt1821682,Traffic,2011,Malayalam,"Action, Adventure, Mystery",Rajesh Pillai,8.1,4271,0.443,genre: Action/Adventure/Mystery; same era
3,imdb:tt1305797,Enthiran,2010,Tamil,"Action, Science Fiction, Thriller",Shankar Shanmugham,7.1,25248,0.442,genre: Action/Science Fiction/Thriller; same era
4,tmdb:49026,The Dark Knight Rises,2012,English,"Action, Crime, Drama, Thriller",Christopher Nolan,7.6,9106,0.435,genre: Action/Thriller; language: English; dir...
5,imdb:tt5947332,Cryptic Road,2016,Hindi,"Mystery, Science Fiction, Thriller","Moosa Saleem, Hardik Kapadia",7.3,23,0.433,genre: Mystery/Science Fiction/Thriller
6,tmdb:1124,The Prestige,2006,English,"Drama, Mystery, Thriller",Christopher Nolan,8.0,4391,0.433,genre: Mystery/Thriller; language: English; di...
7,imdb:tt3518888,Poovarasam Peepee,2014,Tamil,"Adventure, Mystery, Thriller",Halitha Shameem,7.6,74,0.428,genre: Adventure/Mystery/Thriller
8,imdb:tt4265964,Badshahi Angti,2014,Bengali,"Adventure, Mystery, Thriller",Sandip Ray,6.6,977,0.423,genre: Adventure/Mystery/Thriller
9,imdb:tt2123894,An Human,2011,Tamil,"Drama, Mystery, Science Fiction",Koushik Kr,6.9,22,0.422,genre: Mystery/Science Fiction; same era
10,imdb:tt1725795,7 Aum Arivu,2011,Tamil,"Action, Science Fiction, Thriller",A.R. Murugadoss,6.3,10417,0.421,genre: Action/Science Fiction/Thriller; same era


MMR-diversified ranking:


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,tmdb:157336,Interstellar,2014,English,"Adventure, Drama, Science Fiction",Christopher Nolan,8.1,10867,0.472,genre: Adventure/Science Fiction; language: En...
2,tmdb:1124,The Prestige,2006,English,"Drama, Mystery, Thriller",Christopher Nolan,8.0,4391,0.433,genre: Mystery/Thriller; language: English; di...
3,tmdb:49026,The Dark Knight Rises,2012,English,"Action, Crime, Drama, Thriller",Christopher Nolan,7.6,9106,0.435,genre: Action/Thriller; language: English; dir...
4,imdb:tt1821682,Traffic,2011,Malayalam,"Action, Adventure, Mystery",Rajesh Pillai,8.1,4271,0.443,genre: Action/Adventure/Mystery; same era
5,imdb:tt1305797,Enthiran,2010,Tamil,"Action, Science Fiction, Thriller",Shankar Shanmugham,7.1,25248,0.442,genre: Action/Science Fiction/Thriller; same era
6,tmdb:77,Memento,2000,English,"Mystery, Thriller",Christopher Nolan,8.1,4028,0.416,genre: Mystery/Thriller; language: English; di...
7,imdb:tt2123894,An Human,2011,Tamil,"Drama, Mystery, Science Fiction",Koushik Kr,6.9,22,0.422,genre: Mystery/Science Fiction; same era
8,tmdb:76341,Mad Max: Fury Road,2015,English,"Action, Adventure, Science Fiction, Thriller",George Miller,7.2,9427,0.415,genre: Action/Adventure/Science Fiction/Thrill...
9,tmdb:198663,The Maze Runner,2014,English,"Action, Mystery, Science Fiction, Thriller",Wes Ball,7.0,5371,0.411,genre: Action/Mystery/Science Fiction/Thriller...
10,tmdb:24428,The Avengers,2012,English,"Action, Adventure, Science Fiction",Joss Whedon,7.4,11776,0.392,genre: Action/Adventure/Science Fiction; langu...


## 11. Basic evaluation and limitations

The catalogue has no user–item interaction history. Therefore, these checks are **content-based proxy diagnostics**, not proof of user satisfaction or real-world recommendation accuracy.

The metrics below measure genre overlap, language match, and intra-list genre diversity. Because genres are also a model input, genre overlap is partly circular and must be interpreted cautiously.

In [12]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0

def evaluate_one_query(title, year=None, n=10):
    query_idx = rec.find_movie(title, year)
    query_row = rec.m.iloc[query_idx]
    results = rec.recommend(
        title=title, year=year, n=n,
        language_mode="ignore", diversify=False, verbose=False
    )

    if results.empty:
        return {"Query": title, "Returned": 0,
                "Genre Precision@10": np.nan,
                "Language Match@10": np.nan,
                "Genre Diversity": np.nan}

    rec_indices = [
        int(np.flatnonzero(rec.m["movie_id"].to_numpy() == movie_id)[0])
        for movie_id in results["Movie_ID"]
    ]

    genre_scores = [
        jaccard(query_row.genres, rec.m.iloc[j].genres)
        for j in rec_indices
    ]

    query_languages = set(query_row.languages)
    language_scores = [
        bool(query_languages & set(rec.m.iloc[j].languages))
        for j in rec_indices
    ]

    pairwise_distance = [
        1 - jaccard(rec.m.iloc[a].genres, rec.m.iloc[b].genres)
        for pos, a in enumerate(rec_indices)
        for b in rec_indices[pos + 1:]
    ]

    return {
        "Query": title,
        "Returned": len(results),
        "Genre Precision@10": float(np.mean([x >= 0.5 for x in genre_scores])),
        "Language Match@10": float(np.mean(language_scores)),
        "Genre Diversity": float(np.mean(pairwise_distance)) if pairwise_distance else np.nan
    }

proxy_queries = [
    ("Avatar", 2009),
    ("Inception", 2010),
    ("The Dark Knight", 2008),
    ("3 Idiots", None),
    ("Ahetuk", 2015),
    ("Classmates", 2006),
]

proxy_rows = []
for title, year in proxy_queries:
    try:
        proxy_rows.append(evaluate_one_query(title, year))
    except KeyError as exc:
        print("Skipped:", exc)

proxy_results = pd.DataFrame(proxy_rows)
display(proxy_results)

,Query,Returned,Genre Precision@10,Language Match@10,Genre Diversity
0,Avatar,10,0.9,0.7,0.436667
1,Inception,10,0.6,0.3,0.675556
2,The Dark Knight,10,1.0,0.2,0.138889
3,3 Idiots,10,1.0,0.4,0.100000
4,Ahetuk,10,1.0,0.0,0.066667
5,Classmates,10,1.0,0.1,0.229630


## 12. Feature ablation

Ablation checks how recommendations change when one feature block is removed. This is diagnostic only; it does not establish ground-truth relevance without human or behavioural labels.

In [13]:
ablation_queries = ["Inception", "3 Idiots"]
ablation_rows = []

for query_title in ablation_queries:
    for removed_block in [None, "genres", "keywords", "text", "cast", "director", "title"]:
        weights = dict(DEFAULT_WEIGHTS)
        label = "All blocks"
        if removed_block is not None:
            weights[removed_block] = 0.0
            label = f"Without {removed_block}"

        try:
            result = rec.recommend(
                title=query_title,
                n=5,
                language_mode="ignore",
                diversify=False,
                weights=weights,
                quality_weight=0.1,
                verbose=False
            )
            ablation_rows.append({
                "Query": query_title,
                "Configuration": label,
                "Recommendations returned": len(result),
                "Titles": " | ".join(result["Title"].tolist())
            })
        except KeyError as exc:
            print("Skipped:", exc)

display(pd.DataFrame(ablation_rows))

,Query,Configuration,Recommendations returned,Titles
0,Inception,All blocks,5,Interstellar | Traffic | Enthiran | The Dark K...
1,Inception,Without genres,5,The Dark Knight Rises | Batman Begins | The Da...
2,Inception,Without keywords,5,Interstellar | The Prestige | The Dark Knight ...
3,Inception,Without text,5,Interstellar | The Dark Knight Rises | The Pre...
4,Inception,Without cast,5,Interstellar | Traffic | Enthiran | Cryptic Ro...
5,Inception,Without director,5,Traffic | Enthiran | Cryptic Road | Poovarasam...
6,Inception,Without title,5,Interstellar | Traffic | Enthiran | Cryptic Ro...
7,3 Idiots,All blocks,5,Munna Bhai M.B.B.S. | Mumbai Xpress | Rocket S...
8,3 Idiots,Without genres,5,PK | Munna Bhai M.B.B.S. | Lage Raho Munna Bha...
9,3 Idiots,Without keywords,5,Munna Bhai M.B.B.S. | Mumbai Xpress | Rocket S...


## 13. Export the modelling catalogue

This exports the prepared catalogue used by the notebook. It does not export fitted vectorizers or sparse matrices; those can be packaged separately when the backend is implemented.

In [14]:
export_path = Path("movie_project/data/final/movies_model_ready.csv")
export = movies.copy()

for column in ["genres", "keywords", "cast_display", "directors_list", "languages",
               "cast_tok", "director_tok"]:
    export[column] = export[column].apply(
        lambda values: "|".join(values) if isinstance(values, list) else values
    )

export.to_csv(export_path, index=False)
print("Saved:", export_path.resolve())
print("Export shape:", export.shape)

## 14. Results and limitations

### What this model does
- Uses six independent TF-IDF feature blocks (genres, keywords, plot text, cast, director, title) plus a year-proximity signal.
- Ranks candidates using weighted cosine similarity with a `missing_penalty` for blocks a candidate lacks.
- Supports language preference and strict language filtering.
- Uses a source-specific Bayesian rating percentile (vote-floored prior) as a quality signal.
- Supports MMR diversification, movie-to-movie and preference-based (cold-start) recommendations.
- Uses v4 enriched Indian genres, directors and cast when available.

### Current data limitations
- Indian records have **0% keyword and plot coverage**; their similarity rests on genre, cast, director, title and year.
- Some Indian records still have incomplete genres, directors or cast.
- Only 21,146 of 50,403 records are eligible recommendation candidates (`recsys_candidate_default`).
- Content-based proxy metrics (genre overlap, language match) are partly circular and are not user-satisfaction measures. Human evaluation or interaction data is required for stronger claims.

### Reproducibility
Record the catalogue filename, date, package versions (printed in section 1), feature weights, `missing_penalty`, `MIN_VOTES_PRIOR` and the baseline query list when reporting results. Re-run the baseline after any change to weights or preprocessing.


In [15]:
exported = pd.read_csv(
    "movie_project/data/final/movies_model_ready.csv",
    low_memory=False
)

print("Shape:", exported.shape)
print("Duplicate IDs:", exported["movie_id"].duplicated().sum())

print("\nColumns:")
print(exported.columns.tolist())

Shape: (50403, 68)
Duplicate IDs: 0

Columns:
['movie_id', 'source_primary', 'tmdb_id', 'imdb_id', 'title', 'original_title', 'year', 'release_date', 'release_type', 'runtime', 'genres_original', 'genres', 'original_language_code', 'original_language', 'primary_language', 'primary_language_basis', 'languages_available', 'overview', 'tagline', 'keywords', 'director', 'directors', 'cast_top10', 'cast_top10_ids', 'rating', 'votes', 'rating_source', 'imdb_rating', 'imdb_votes', 'popularity', 'budget', 'revenue', 'production_countries', 'status', 'poster_path', 'homepage', 'linked_ids', 'link_confidence', 'data_flags', 'has_genre', 'has_overview', 'has_keywords', 'has_cast', 'has_director', 'content_blocks', 'recsys_query_ok', 'recsys_candidate_default', 'needs_enrichment', 'genres_enriched', 'imdb_directors_clean', 'imdb_cast', 'genres_recommender', 'directors_recommender', 'cast_recommender', 'source', 'cast_display', 'directors_list', 'languages', 'text', 'keyword_text', 'cast_tok', 'dir

In [16]:

baseline_queries = [
    ("Q01", "Avatar", 2009, "English"),
    ("Q02", "Inception", 2010, "English"),
    ("Q03", "The Dark Knight", 2008, "English"),
    ("Q04", "The Shawshank Redemption", 1994, "English"),
    ("Q05", "The Matrix", 1999, "English"),
    ("Q06", "The Lord of the Rings: The Fellowship of the Ring", 2001, "English"),
    ("Q07", "Titanic", 1997, "English"),
    ("Q08", "Interstellar", 2014, "English"),
    ("Q09", "The Prestige", 2006, "English"),
    ("Q10", "Mad Max: Fury Road", 2015, "English"),
    ("Q11", "3 Idiots", 2009, "Hindi"),
    ("Q12", "Ahetuk", 2015, "Assamese"),
    ("Q13", "Classmates", 2006, "Malayalam"),
    ("Q14", "Dangal", 2016, "Hindi"),
    ("Q15", "Andhadhun", 2018, "Hindi"),
    ("Q16", "Drishyam", 2013, "Malayalam"),
    ("Q17", "Kaithi", 2019, "Tamil"),
    ("Q18", "Super Deluxe", 2019, "Tamil"),
    ("Q19", "Kumbalangi Nights", 2019, "Malayalam"),
    ("Q20", "Pather Panchali", 1955, "Bengali"),
]

baseline_rows = []

for qid, title, year, language in baseline_queries:

    recommendations = rec.recommend(
        title=title,
        year=year,
        source_language=language,
        n=5,
        language_mode="ignore",
        diversify=False,
        verbose=False
    )

    for rank, (_, movie) in enumerate(
        recommendations.iterrows(), start=1
    ):
        baseline_rows.append({
            "Query_ID": qid,
            "Query_Title": title,
            "Query_Year": year,
            "Query_Language": language,
            "Rank": rank,
            "Recommended_Title": movie["Title"],
            "Recommended_Year": movie["Year"],
            "Recommended_Language": movie["Language"]
        })

baseline_results = pd.DataFrame(baseline_rows)

baseline_results.to_csv(
    "movie_project/data/final/baseline_recommendations.csv",
    index=False
)

print("Total recommendations:", len(baseline_results))
display(baseline_results)


Total recommendations: 100


,Query_ID,Query_Title,Query_Year,Query_Language,Rank,Recommended_Title,Recommended_Year,Recommended_Language
0,Q01,Avatar,2009,English,1,Star Trek Into Darkness,2013,English
1,Q01,Avatar,2009,English,2,Uncharted 2: Among Thieves,2009,Nepali
2,Q01,Avatar,2009,English,3,Star Trek,2009,English
3,Q01,Avatar,2009,English,4,X-Men: Days of Future Past,2014,English
4,Q01,Avatar,2009,English,5,The Abyss,1989,English
...,...,...,...,...,...,...,...,...
95,Q20,Pather Panchali,1955,Bengali,1,Mayura,1975,Kannada
96,Q20,Pather Panchali,1955,Bengali,2,Shatranj Ke Khilari,1977,"Hindi, Urdu"
97,Q20,Pather Panchali,1955,Bengali,3,Avvaiyyar,1953,Tamil
98,Q20,Pather Panchali,1955,Bengali,4,Chhatrapati Shivaji,1952,Hindi


In [17]:
indian_queries = baseline_results[
    baseline_results["Query_ID"].isin([
        "Q11", "Q12", "Q13", "Q14", "Q15",
        "Q16", "Q17", "Q18", "Q19", "Q20"
    ])
]

for qid, group in indian_queries.groupby("Query_ID", sort=True):
    first = group.iloc[0]

    print(
        f"\n{first['Query_Title']} "
        f"({first['Query_Year']}) — {first['Query_Language']}"
    )

    for _, movie in group.iterrows():
        print(
            f"{movie['Rank']}. {movie['Recommended_Title']} "
            f"({movie['Recommended_Year']}) "
            f"| {movie['Recommended_Language']}"
        )



3 Idiots (2009) — Hindi
1. Munna Bhai M.B.B.S. (2003) | Hindi
2. Mumbai Xpress (2005) | Hindi
3. Rocket Singh: Salesman of the Year (2009) | Hindi
4. Eddelu Manjunatha (2009) | Kannada
5. Gabhricha Paus (2009) | Marathi

Ahetuk (2015) — Assamese
1. Classmates (2015) | Marathi
2. Njan Steve Lopez (2014) | Malayalam
3. Devara Naadalli (2016) | Kannada
4. Beautiful Manasugalu (2017) | Kannada
5. Ayalum Njanum Thammil... (2012) | Malayalam

Classmates (2006) — Malayalam
1. Classmates (2007) | Telugu
2. Shehr-e-Zaat (2012) | Urdu
3. The Lake House (2006) | English
4. Megher Koley Rod (2008) | Bengali
5. Karthik Calling Karthik (2010) | Hindi, Urdu

Dangal (2016) — Hindi
1. Salute (2016) | Urdu
2. Hippocratic (2017) | Malayalam
3. Sajjan Singh Rangroot (2018) | Punjabi
4. Dakuaan Da Munda (2018) | Punjabi
5. Subedar Joginder Singh (2018) | Punjabi, Tamil, Telugu

Andhadhun (2018) — Hindi
1. Badlapur (2015) | Hindi
2. Monpura (2009) | Bengali
3. Secret Superstar (2017) | Hindi
4. Dancer in t

In [18]:

language_tests = [
    ("3 Idiots", 2009, "Hindi"),
    ("Classmates", 2006, "Malayalam"),
    ("Kaithi", 2019, "Tamil"),
    ("Pather Panchali", 1955, "Bengali"),
]

for title, year, language in language_tests:

    print(f"\n{'='*60}")
    print(f"Query: {title} ({language})")

    recommendations = rec.recommend(
        title=title,
        year=year,
        source_language=language,
        n=5,
        language_mode="prefer",
        diversify=False
    )

    display(recommendations)



Query: 3 Idiots (Hindi)
Because you picked: 3 Idiots (2009) · Hindi · Comedy, Drama   |  language_mode=prefer, target=Hindi


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt0374887,Munna Bhai M.B.B.S.,2003,Hindi,"Comedy, Drama",Rajkumar Hirani,8.1,74793,1.378,genre: Comedy/Drama; language: Hindi; director...
2,imdb:tt11167150,Mumbai Xpress,2005,Hindi,Comedy,,6.9,58,1.255,genre: Comedy; language: Hindi
3,imdb:tt1434447,Rocket Singh: Salesman of the Year,2009,Hindi,"Comedy, Drama",Shimit Amin,7.5,19102,1.253,genre: Comedy/Drama; language: Hindi; same era
4,imdb:tt1288638,Dasvidaniya,2008,Hindi,"Comedy, Drama",Shashant Shah,7.8,6081,1.245,genre: Comedy/Drama; language: Hindi; same era
5,imdb:tt2338151,PK,2014,Hindi,"Comedy, Drama, Music",Rajkumar Hirani,8.1,165617,1.244,genre: Comedy/Drama; language: Hindi; director...



Query: Classmates (Malayalam)
Because you picked: Classmates (2006) · Malayalam · Drama, Mystery, Romance   |  language_mode=prefer, target=Malayalam


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt0400186,Chanthupottu,2005,Malayalam,"Comedy, Drama, Romance",Lal Jose,6.6,724,1.191,genre: Drama/Romance; language: Malayalam; dir...
2,imdb:tt7382730,Luca,2019,Malayalam,"Drama, Mystery, Romance",Arun Bose,7.4,1414,1.179,genre: Drama/Mystery/Romance; language: Malayalam
3,imdb:tt2146836,Diamond Necklace,2012,Malayalam,"Drama, Romance",Lal Jose,7.2,1688,1.176,genre: Drama/Romance; language: Malayalam; dir...
4,imdb:tt0255043,Chandranudikkunna Dikhil,1999,Malayalam,Romance,Lal Jose,6.8,249,1.154,genre: Romance; language: Malayalam; director:...
5,imdb:tt2476214,Ayalum Njanum Thammil...,2012,Malayalam,"Drama, Romance, Thriller",Lal Jose,8.0,2579,1.149,genre: Drama/Romance; language: Malayalam; dir...



Query: Kaithi (Tamil)
Because you picked: Kaithi (2019) · Tamil · Action, Crime, Thriller   |  language_mode=prefer, target=Tamil


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt10579952,Master,2021,Tamil,"Action, Thriller",Lokesh Kanagaraj,7.0,51930,1.300,genre: Action/Thriller; language: Tamil; direc...
2,imdb:tt5559528,Maanagaram,2017,Tamil,"Action, Thriller",Lokesh Kanagaraj,8.1,4474,1.277,genre: Action/Thriller; language: Tamil; direc...
3,imdb:tt7060460,Theeran Adhigaram Ondru,2017,Tamil,"Action, Crime, Thriller","H. Vinoth, Hariharakrishnan Ramalingam",8.2,10814,1.261,genre: Action/Crime/Thriller; language: Tamil;...
4,imdb:tt8042132,Kaalidas,2019,Tamil,"Action, Crime, Thriller",Sri Senthil,6.9,978,1.249,genre: Action/Crime/Thriller; language: Tamil;...
5,imdb:tt7060344,Raatchasan,2018,Tamil,"Action, Crime, Thriller",Ram Kumar,8.5,27654,1.247,genre: Action/Crime/Thriller; language: Tamil;...



Query: Pather Panchali (Bengali)
Because you picked: Pather Panchali (1955) · Bengali · Drama, History   |  language_mode=prefer, target=Bengali


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt0319836,Matir Moina,2002,Bengali,"Drama, History",Tareque Masud,8.6,3840,1.147,genre: Drama/History; language: Bengali
2,imdb:tt4948632,Rajkahini,2015,Bengali,"Drama, History",Srijit Mukherji,7.3,1482,1.142,genre: Drama/History; language: Bengali
3,imdb:tt5537458,Shankhachil,2016,Bengali,"Drama, History",Goutam Ghose,7.3,366,1.141,genre: Drama/History; language: Bengali
4,imdb:tt4658786,Kadambari,2015,Bengali,"Drama, History",Suman Ghosh,7.1,297,1.139,genre: Drama/History; language: Bengali
5,imdb:tt7421526,Khacha,2017,Bengali,"Drama, History",Akram Khan,7.3,35,1.134,genre: Drama/History; language: Bengali


In [19]:
availability_report = []

model_movies = rec.m

for feature, availability in rec.block_available.items():

    availability = np.asarray(availability, dtype=bool)

    for source in ["Indian", "TMDB"]:

        mask = model_movies["source"].eq(source).to_numpy()

        total = mask.sum()
        available = (availability & mask).sum()

        availability_report.append({
            "Source": source,
            "Feature": feature,
            "Available": int(available),
            "Total": int(total),
            "Coverage_%": round(available / total * 100, 2)
        })

display(pd.DataFrame(availability_report))

,Source,Feature,Available,Total,Coverage_%
0,Indian,genres,37647,45600,82.56
1,TMDB,genres,4775,4803,99.42
2,Indian,keywords,0,45600,0.00
3,TMDB,keywords,4379,4803,91.17
4,Indian,text,0,45600,0.00
5,TMDB,text,4799,4803,99.92
6,Indian,cast,40539,45600,88.90
7,TMDB,cast,4760,4803,99.10
8,Indian,director,39712,45600,87.09
9,TMDB,director,4773,4803,99.38


## 15. Diagnostics

Score decomposition for any query/candidate pair, and regression checks for this revision.

In [20]:
def explain_pair(query_title, candidate_title, query_year=None, candidate_year=None, rec=rec):
    """Show how one candidate's score is built. Replaces the old Kikori-specific debugging cells."""
    qi = rec.find_movie(query_title, query_year)
    ci = rec.find_movie(candidate_title, candidate_year)
    sims = rec.query_sims(qi)
    rows = [{"block": b, "similarity": round(float(s[ci]), 4), "weight": rec.weights.get(b, 0),
             "candidate_has_data": bool(rec.block_available[b][ci])} for b, s in sims.items()]
    display(pd.DataFrame(rows))
    print("Combined content score:", round(float(rec.combine(sims)[ci]), 4))
    print("Quality contribution (x0.1):", round(0.1 * rec.quality[ci], 4))

explain_pair("Kaithi", "Kikori")   # was 0.599 before the missing_penalty fix


,block,similarity,weight,candidate_has_data
0,genres,0.6564,3.00,True
1,cast,0.0000,1.50,False
2,director,0.0000,1.00,False
3,title,0.0000,0.50,True
4,year,0.7788,0.75,True


Combined content score: 0.4642
Quality contribution (x0.1): 0.0859


In [21]:
# Regression checks for the fixes in this revision
# 1. Cross-source genre sharing now possible
ind = {g for gs in movies.loc[movies.source == "Indian", "genres"] for g in gs}
tm  = {g for gs in movies.loc[movies.source == "TMDB", "genres"] for g in gs}
print(f"Genres shared by Indian and TMDB: {len(ind & tm)} of {len(ind | tm)}")

# 2. Titles with no title vector (previously 63 TMDB films incl. Up, Nine, The One)
print("Movies without a title vector:", int((~rec.block_available["title"]).sum()))

# 3. Sparse-metadata check: median metadata blocks per top-10 result for an Indian query
res = rec.recommend("Kaithi", year=2019, source_language="Tamil", n=10, language_mode="ignore", verbose=False)
print(res[["Title", "Year", "Language", "Score"]])


Genres shared by Indian and TMDB: 18 of 23
Movies without a title vector: 0
                      Title  Year   Language  Score
1                    Master  2021      Tamil  0.700
2                Maanagaram  2017      Tamil  0.677
3                      Safe  2019  Malayalam  0.669
4   Theeran Adhigaram Ondru  2017      Tamil  0.661
5                Rakshasudu  2019     Telugu  0.655
6                Jallikattu  2019  Malayalam  0.653
7                  Kaalidas  2019      Tamil  0.649
8           Arjun Suravaram  2019     Telugu  0.648
9                Raatchasan  2018      Tamil  0.647
10             Thani Oruvan  2015      Tamil  0.645


In [22]:
test_movies = [
    "Interstellar",
    "3 Idiots",
    "Kaithi",
    "Pather Panchali",
    "Andhadhun"
]

for movie in test_movies:
    print("\n" + "=" * 70)
    print(f"RECOMMENDATIONS FOR: {movie}")
    print("=" * 70)

    display(rec.recommend(movie, n=10))


RECOMMENDATIONS FOR: Interstellar
Because you picked: Interstellar (2014) · English · Adventure, Drama, Science Fiction   |  language_mode=balanced, target=English


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,tmdb:286217,The Martian,2015,"English, Mandarin","Adventure, Drama, Science Fiction",Ridley Scott,7.6,7268,0.673,genre: Adventure/Drama/Science Fiction; langua...
2,tmdb:27205,Inception,2010,"English, Japanese, French","Action, Adventure, Mystery, Science Fiction, T...",Christopher Nolan,8.1,13752,0.622,genre: Adventure/Science Fiction; language: En...
3,tmdb:118340,Guardians of the Galaxy,2014,English,"Action, Adventure, Science Fiction",James Gunn,7.9,9742,0.618,genre: Adventure/Science Fiction; language: En...
4,tmdb:54138,Star Trek Into Darkness,2013,English,"Action, Adventure, Science Fiction",J.J. Abrams,7.4,4418,0.610,genre: Adventure/Science Fiction; language: En...
5,tmdb:13001,Stargate: The Ark of Truth,2008,English,"Adventure, Science Fiction",Robert C. Cooper,6.9,140,0.608,genre: Adventure/Science Fiction; language: En...
6,tmdb:13475,Star Trek,2009,English,"Action, Adventure, Science Fiction",J.J. Abrams,7.4,4518,0.598,genre: Adventure/Science Fiction; language: En...
7,tmdb:68726,Pacific Rim,2013,English,"Action, Adventure, Science Fiction",Guillermo del Toro,6.7,4794,0.597,genre: Adventure/Science Fiction; language: En...
8,tmdb:122906,About Time,2013,English,"Comedy, Drama, Science Fiction",Richard Curtis,7.8,2067,0.587,genre: Drama/Science Fiction; language: Englis...
9,tmdb:101299,The Hunger Games: Catching Fire,2013,English,"Action, Adventure, Science Fiction",Francis Lawrence,7.4,6495,0.587,genre: Adventure/Science Fiction; language: En...
10,tmdb:100402,Captain America: The Winter Soldier,2014,English,"Action, Adventure, Science Fiction","Anthony Russo, Joe Russo",7.6,5764,0.585,genre: Adventure/Science Fiction; language: En...



RECOMMENDATIONS FOR: 3 Idiots
Because you picked: 3 Idiots (2009) · Hindi · Comedy, Drama   |  language_mode=balanced, target=Hindi


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt0374887,Munna Bhai M.B.B.S.,2003,Hindi,"Comedy, Drama",Rajkumar Hirani,8.1,74793,0.928,genre: Comedy/Drama; language: Hindi; director...
2,imdb:tt11167150,Mumbai Xpress,2005,Hindi,Comedy,,6.9,58,0.805,genre: Comedy; language: Hindi
3,imdb:tt1434447,Rocket Singh: Salesman of the Year,2009,Hindi,"Comedy, Drama",Shimit Amin,7.5,19102,0.803,genre: Comedy/Drama; language: Hindi; same era
4,imdb:tt1288638,Dasvidaniya,2008,Hindi,"Comedy, Drama",Shashant Shah,7.8,6081,0.795,genre: Comedy/Drama; language: Hindi; same era
5,imdb:tt2338151,PK,2014,Hindi,"Comedy, Drama, Music",Rajkumar Hirani,8.1,165617,0.794,genre: Comedy/Drama; language: Hindi; director...
6,imdb:tt1447508,Peepli [Live],2010,Hindi,"Comedy, Drama","Anusha Rizvi, Mahmood Farooqui",7.4,11885,0.794,genre: Comedy/Drama; language: Hindi; same era
7,imdb:tt1629391,Tere Bin Laden,2010,"Hindi, Punjabi, Urdu","Comedy, Drama",Abhishek Sharma,7.2,10196,0.793,genre: Comedy/Drama; language: Hindi; same era
8,imdb:tt5342784,Le Lotta,2017,Hindi,"Comedy, Drama",Manish Mitra,6.5,15,0.792,genre: Comedy/Drama; language: Hindi
9,imdb:tt1221139,Welcome to Sajjanpur,2008,Hindi,"Comedy, Drama",Shyam Benegal,6.9,3072,0.790,genre: Comedy/Drama; language: Hindi; same era
10,imdb:tt1740670,Dus Tola,2010,Hindi,"Comedy, Drama",Ajoy Varma Raja,7.0,300,0.790,genre: Comedy/Drama; language: Hindi; same era



RECOMMENDATIONS FOR: Kaithi
Because you picked: Kaithi (2019) · Tamil · Action, Crime, Thriller   |  language_mode=balanced, target=Tamil


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt10579952,Master,2021,Tamil,"Action, Thriller",Lokesh Kanagaraj,7.0,51930,0.850,genre: Action/Thriller; language: Tamil; direc...
2,imdb:tt5559528,Maanagaram,2017,Tamil,"Action, Thriller",Lokesh Kanagaraj,8.1,4474,0.827,genre: Action/Thriller; language: Tamil; direc...
3,imdb:tt7060460,Theeran Adhigaram Ondru,2017,Tamil,"Action, Crime, Thriller","H. Vinoth, Hariharakrishnan Ramalingam",8.2,10814,0.811,genre: Action/Crime/Thriller; language: Tamil;...
4,imdb:tt8042132,Kaalidas,2019,Tamil,"Action, Crime, Thriller",Sri Senthil,6.9,978,0.799,genre: Action/Crime/Thriller; language: Tamil;...
5,imdb:tt7060344,Raatchasan,2018,Tamil,"Action, Crime, Thriller",Ram Kumar,8.5,27654,0.797,genre: Action/Crime/Thriller; language: Tamil;...
6,imdb:tt4987556,Thani Oruvan,2015,Tamil,"Action, Crime, Thriller",Mohan Raja,8.4,17172,0.795,genre: Action/Crime/Thriller; language: Tamil;...
7,imdb:tt7179784,Irumbu Thirai,2018,Tamil,"Action, Crime, Thriller",P.S. Mithran,7.7,3882,0.795,genre: Action/Crime/Thriller; language: Tamil;...
8,imdb:tt7179714,Iravukku Aayiram Kangal,2018,Tamil,"Action, Crime, Thriller",Mu. Maran,7.3,1059,0.793,genre: Action/Crime/Thriller; language: Tamil;...
9,imdb:tt13330846,Patra Vaitha Nerupondru,2020,Tamil,"Action, Crime, Thriller",Vinoth Rajendran,7.9,91,0.792,genre: Action/Crime/Thriller; language: Tamil;...
10,imdb:tt6747242,8 Thottakkal,2017,Tamil,"Action, Crime, Thriller","Sri Ganesh, Sundaramurthy Ks, T Siva, Dilip Su...",7.6,1622,0.786,genre: Action/Crime/Thriller; language: Tamil;...



RECOMMENDATIONS FOR: Pather Panchali
Because you picked: Pather Panchali (1955) · Bengali · Drama, History   |  language_mode=balanced, target=Bengali


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt0319836,Matir Moina,2002,Bengali,"Drama, History",Tareque Masud,8.6,3840,0.697,genre: Drama/History; language: Bengali
2,imdb:tt4948632,Rajkahini,2015,Bengali,"Drama, History",Srijit Mukherji,7.3,1482,0.692,genre: Drama/History; language: Bengali
3,imdb:tt5537458,Shankhachil,2016,Bengali,"Drama, History",Goutam Ghose,7.3,366,0.691,genre: Drama/History; language: Bengali
4,imdb:tt4658786,Kadambari,2015,Bengali,"Drama, History",Suman Ghosh,7.1,297,0.689,genre: Drama/History; language: Bengali
5,imdb:tt7421526,Khacha,2017,Bengali,"Drama, History",Akram Khan,7.3,35,0.684,genre: Drama/History; language: Bengali
6,imdb:tt1042463,Adhiar,2003,Bengali,"Drama, History",Saidul Anam Tutul,6.4,15,0.673,genre: Drama/History; language: Bengali
7,imdb:tt0048956,Aparajito,1956,Bengali,Drama,Satyajit Ray,8.4,11954,0.671,genre: Drama; language: Bengali; director: Sat...
8,imdb:tt1416753,Mayura,1975,Kannada,"Drama, History",,8.2,236,0.670,genre: Drama/History
9,imdb:tt5077742,Ora 11 Jon,1972,Bengali,"Drama, History, Romance",Chashi Nazrul Islam,8.8,353,0.669,genre: Drama/History; language: Bengali
10,imdb:tt5782250,Nawab Sirajuddaula,1989,Bengali,History,Pradeep Dey,7.5,12,0.664,genre: History; language: Bengali



RECOMMENDATIONS FOR: Andhadhun
Because you picked: Andhadhun (2018) · Hindi · Crime, Drama, Music   |  language_mode=balanced, target=Hindi


,Movie_ID,Title,Year,Language,Genres,Director,Rating,Votes,Score,Why
1,imdb:tt3678782,Badlapur,2015,Hindi,"Action, Crime, Drama",Sriram Raghavan,7.4,22033,0.756,genre: Crime/Drama; language: Hindi; director:...
2,imdb:tt6108090,Secret Superstar,2017,Hindi,"Drama, Music",Advait Chandan,7.8,21334,0.727,genre: Drama/Music; language: Hindi; cast: Kab...
3,imdb:tt0106333,Baazigar,1993,Hindi,"Crime, Drama, Music","Abbas Alibhai Burmawalla, Mastan Alibhai Burma...",7.7,26193,0.707,genre: Crime/Drama/Music; language: Hindi
4,imdb:tt0080322,Aakrosh,1980,Hindi,"Crime, Drama, Music",Govind Nihalani,7.9,864,0.698,genre: Crime/Drama/Music; language: Hindi
5,imdb:tt0061842,Jewel Thief,1967,Hindi,"Crime, Drama, Music",Vijay Anand,8.0,1748,0.695,genre: Crime/Drama/Music; language: Hindi
6,imdb:tt0053980,Kala Bazar,1960,Hindi,"Crime, Drama, Music",Vijay Anand,7.4,256,0.691,genre: Crime/Drama/Music; language: Hindi
7,imdb:tt0041123,Andaz,1949,Hindi,"Crime, Drama, Music",Mehboob Khan,7.3,483,0.691,genre: Crime/Drama/Music; language: Hindi
8,imdb:tt0043307,Baazi,1951,Hindi,"Crime, Drama, Music",Guru Dutt,7.3,321,0.690,genre: Crime/Drama/Music; language: Hindi
9,imdb:tt12969078,Atkan Chatkan,2020,Hindi,"Drama, Music",Shiv Hare,7.9,1727,0.688,genre: Drama/Music; language: Hindi; same era
10,imdb:tt10230436,Yeh Ballet,2020,Hindi,"Drama, Music",Sooni Taraporevala,7.6,1096,0.687,genre: Drama/Music; language: Hindi; same era
